# Inventory of the 292 CPI Generic Items (INPC 2024)
## Exploratory Analysis and Linkage with the G-CPI Project

**Source file:** `INPC Inventory – 292 Generic Items.xlsx`  
**Official source:** INEGI — Update of the CPI Basket and Weights 2024 (published August 22, 2024)  
**Sub-generic weights:** ENIGH E_AVPR, first half of November 2023

---
Run **Kernel → Restart & Run All** to execute all cells at once.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1 — IMPORTS AND DATA LOAD
# ═══════════════════════════════════════════════════════════════════════════════
import glob, pandas as pd, numpy as np
import matplotlib.pyplot as plt, openpyxl
from pathlib import Path

OUT_DIR = Path('outputs')
OUT_DIR.mkdir(exist_ok=True)

# Full path to the source file
FILENAME = r'D:\Downloads\Inventario INPC_292 genéricos.xlsx'

wb   = openpyxl.load_workbook(FILENAME, read_only=True)
ws   = wb.active
rows = list(ws.iter_rows(values_only=True))

# Headers are on row 4 (index 3); data starts on row 5
headers = rows[3]
data    = [r for r in rows[4:] if any(x is not None for x in r)]
df      = pd.DataFrame(data, columns=headers)
df['num'] = pd.to_numeric(df['No. de gen.'], errors='coerce')

print(f'✅ Loaded: {len(df):,} rows — {df["No. de gen."].nunique()} generics')
df.head(3)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2 — STATUS OF THE 292 GENERICS vs. THE 2018 BASKET
# ═══════════════════════════════════════════════════════════════════════════════
sit = df.drop_duplicates('No. de gen.')['Situación del genérico'].value_counts()

# Translate status labels for display
STATUS_EN = {
    'Igual':       'Unchanged',
    'Desagregado': 'Disaggregated',
    'Fusionado':   'Merged',
}

print('═' * 56)
print('  STATUS OF THE 292 GENERICS vs. THE 2018 BASKET')
print('═' * 56)
for k, v in sit.items():
    pct   = v / 292 * 100
    label = STATUS_EN.get(k, k)
    print(f'  {label:<18} {v:>4} generics  ({pct:.1f}%)')
print('─' * 56)
print(f'  TOTAL              {sit.sum():>4}')
print()
print('Context: the 2018 basket had 299 generics → 2024 has 292')
print('  • 261 remained Unchanged')
print('  •  25 were Disaggregated (e.g. Streaming, Energy drinks, Cilantro)')
print('  •   6 were Merged (e.g. Shrimp + Other seafood)')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 3 — MAPPING TO COICOP 2018 DIVISIONS
# ═══════════════════════════════════════════════════════════════════════════════
div_manual = {
    **{i: '01' for i in range(1,   100)},   # Food & non-alcoholic beverages
    **{i: '02' for i in range(100, 110)},   # Alcoholic beverages & tobacco
    **{i: '03' for i in range(110, 138)},   # Clothing & footwear
    **{i: '04' for i in range(138, 147)},   # Housing
    **{i: '05' for i in range(147, 184)},   # Furniture & household goods
    **{i: '06' for i in range(184, 206)},   # Health
    **{i: '07' for i in range(206, 228)},   # Transport
    271: '07',                              # Car insurance → Transport
    **{i: '08' for i in range(228, 238)},   # Communication
    **{i: '09' for i in range(238, 256)},   # Recreation & culture
    **{i: '10' for i in range(256, 263)},   # Education
    **{i: '11' for i in range(263, 271)},   # Restaurants & hotels
    **{i: '12' for i in [287,288,289,290,291,292]},  # Miscellaneous goods & services
    **{i: '13' for i in range(272, 287)},   # Personal care
}

DIV_NAMES = {
    '01': 'Food & non-alcoholic beverages',
    '02': 'Alcoholic beverages & tobacco',
    '03': 'Clothing & footwear',
    '04': 'Housing',
    '05': 'Furniture & household goods',
    '06': 'Health',
    '07': 'Transport',
    '08': 'Communication',
    '09': 'Recreation & culture',
    '10': 'Education',
    '11': 'Restaurants & hotels',
    '12': 'Miscellaneous goods & services',
    '13': 'Personal care',
}

df['ccif_div']  = df['num'].map(div_manual)
uniq            = df.drop_duplicates('No. de gen.').copy()
sin_div         = uniq['ccif_div'].isna().sum()

por_div = (
    uniq.groupby('ccif_div')
    .agg(n_genericos=('No. de gen.', 'count'))
    .reset_index()
)
por_div['div_name'] = por_div['ccif_div'].map(DIV_NAMES)
por_div = por_div.sort_values('ccif_div')

print('═' * 66)
print('  GENERICS BY COICOP DIVISION')
print('═' * 66)
for _, r in por_div.iterrows():
    bar = '█' * r['n_genericos']
    print(f"  {r['ccif_div']}  {r['div_name']:<36} {r['n_genericos']:>3}  {bar}")
print('─' * 66)
print(f"  TOTAL{' ' * 39} {por_div['n_genericos'].sum():>3}")
print(f"  Unassigned: {sin_div}")

por_div.to_csv(OUT_DIR / 'inv292_by_division.csv', index=False)
print('\n✅ Saved: outputs/inv292_by_division.csv')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4 — SUB-GENERICS WITH INTERNAL WEIGHTING
# ═══════════════════════════════════════════════════════════════════════════════
POND_COL = 'Ponderación Subgenéricos (ENIGH E_AVPR 1q nov 23)'
df['weight_subgen'] = pd.to_numeric(df[POND_COL], errors='coerce')

subgen = df[df['weight_subgen'].notna()].copy()
subgen = subgen[['No. de gen.', 'Nombre del Genérico',
                  'Subgenéricos', 'weight_subgen', 'ccif_div']].copy()
subgen['div_name'] = subgen['ccif_div'].map(DIV_NAMES)

print(f'Total weighted sub-generics  : {len(subgen)}')
print(f'Generics that contain them   : {subgen["No. de gen."].nunique()}')
print()

for gen_num in subgen['No. de gen.'].unique():
    sub    = subgen[subgen['No. de gen.'] == gen_num]
    nombre = sub['Nombre del Genérico'].iloc[0]
    div    = sub['ccif_div'].iloc[0]
    print(f'  [{gen_num}] {nombre}  (Div {div} — {DIV_NAMES.get(div, "")})')
    for _, r in sub.iterrows():
        bar = '▓' * int(r['weight_subgen'] / 5)
        print(f'       {r["Subgenéricos"]:<42} {r["weight_subgen"]:>6.2f}%  {bar}')
    print()

subgen.to_csv(OUT_DIR / 'inv292_weighted_subgenerics.csv', index=False)
print('✅ Saved: outputs/inv292_weighted_subgenerics.csv')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5 — GENERICS WITH A DEMOGRAPHIC OR GENDER PROFILE
# ═══════════════════════════════════════════════════════════════════════════════
nombres_uniq = df.drop_duplicates('No. de gen.')[
    ['No. de gen.', 'Nombre del Genérico', 'ccif_div']].copy()

# Keywords that signal a female, male, or child profile (Spanish item names)
kw_woman = ['mujer', 'femenin', 'niña', 'faldas', 'vestido', 'blusas',
             'medias', 'pantimedias', 'sanitarias', 'maquillaje',
             'sala de belleza', 'embarazo']
kw_man   = ['hombre', 'masculin', 'camisa', 'traje', 'afeitar',
             'navajas', 'barbacoa', 'corte de cabello']
kw_child = ['niño', 'niña', 'bebé', 'escolar', 'guardería',
             'infantil', 'preescolar', 'primaria', 'secundaria']

def classify(name):
    n = name.lower()
    flags = []
    if any(k in n for k in kw_woman): flags.append('👩 Woman')
    if any(k in n for k in kw_man):   flags.append('👨 Man')
    if any(k in n for k in kw_child): flags.append('👶 Child')
    return ', '.join(flags) if flags else 'General'

nombres_uniq['demographic'] = nombres_uniq['Nombre del Genérico'].apply(classify)
especificos = nombres_uniq[nombres_uniq['demographic'] != 'General'].copy()
especificos['div_name'] = especificos['ccif_div'].map(DIV_NAMES)

print('═' * 72)
print('  GENERICS WITH AN EXPLICIT DEMOGRAPHIC PROFILE IN THEIR NAME')
print('═' * 72)
print(f'  Total: {len(especificos)} out of 292 generics\n')
for _, r in especificos.sort_values('ccif_div').iterrows():
    print(f"  {r['No. de gen.']}  {r['Nombre del Genérico']:<52} {r['demographic']}")

print('\n  Summary by category:')
for cat, cnt in especificos['demographic'].value_counts().items():
    print(f'    {cat}: {cnt} generics')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 6 — CHART: GENERICS BY DIVISION AND STATUS
# ═══════════════════════════════════════════════════════════════════════════════
# Translate status values for chart labels
uniq['status_en'] = uniq['Situación del genérico'].map(STATUS_EN)

sit_div = (
    uniq.groupby(['ccif_div', 'status_en'])
    .size().reset_index(name='n')
    .pivot_table(index='ccif_div', columns='status_en',
                 values='n', fill_value=0)
    .reset_index()
)
sit_div['name'] = sit_div['ccif_div'].map(DIV_NAMES)
sit_div = sit_div.set_index('name')

cols_sit = [c for c in ['Unchanged', 'Disaggregated', 'Merged'] if c in sit_div.columns]
colors   = {'Unchanged': '#4CAF50', 'Disaggregated': '#2196F3', 'Merged': '#FF9800'}

fig, ax = plt.subplots(figsize=(12, 7))
bottom = np.zeros(len(sit_div))
for col in cols_sit:
    vals = sit_div[col].values
    ax.barh(sit_div.index, vals, left=bottom,
            color=colors[col], label=col, alpha=0.85)
    bottom += vals
for i, total in enumerate(bottom):
    ax.text(total + 0.3, i, str(int(total)), va='center', fontsize=9)

ax.set_xlabel('Number of generics', fontsize=11)
ax.set_title('Composition of the INPC 2024 Basket by COICOP Division\n'
             'Status relative to the 2018 basket',
             fontsize=12, fontweight='bold')
ax.legend(title='Status vs. 2018', fontsize=10)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'inv292_generics_by_division.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: outputs/inv292_generics_by_division.png')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 7 — GENERICS IN DIVISIONS WITH THE LARGEST G-CPI DIFFERENTIALS
# ═══════════════════════════════════════════════════════════════════════════════
# Divisions where the expenditure share differs most across household types
divs_high_diff = {
    '11': ('Restaurants & hotels',            'Male-headed 15.7%  vs  Female-headed 10.3%'),
    '13': ('Personal care',                   'Male-headed 11.1%  vs  Female-headed  6.4%'),
    '01': ('Food & non-alcoholic beverages',  'Female-headed 19.0%  vs  Male-headed 13.7%'),
    '07': ('Transport',                       'Couple (M) 20.9%  vs  Female-headed 15.7%'),
    '10': ('Education',                       'Couple (M)  8.9%  vs  Male-headed    3.9%'),
}

# Keywords that flag a gender/demographic item name
kw_gender = ['mujer', 'hombre', 'niño', 'niña', 'bebé',
             'embarazo', 'afeitar', 'belleza', 'sanitaria']

print('═' * 76)
print('  GENERICS IN DIVISIONS WITH THE LARGEST BETWEEN-GROUP G-CPI DIFFERENTIALS')
print('═' * 76)

for div, (name, note) in divs_high_diff.items():
    gen_div = uniq[uniq['ccif_div'] == div].copy()
    print(f'\n  Division {div} — {name}  |  Differential: {note}')
    print(f'  {len(gen_div)} generics:')
    for _, r in gen_div.iterrows():
        tag = '  ◄ GENDER PROFILE' if any(
            k in r['Nombre del Genérico'].lower() for k in kw_gender) else ''
        print(f"    {r['No. de gen.']}  {r['Nombre del Genérico']}{tag}")

# Export the full table
tabla = uniq[['No. de gen.', 'Nombre del Genérico',
              'ccif_div', 'Situación del genérico']].copy()
tabla['div_name']  = tabla['ccif_div'].map(DIV_NAMES)
tabla['status_en'] = tabla['Situación del genérico'].map(STATUS_EN)
tabla.to_csv(OUT_DIR / 'inv292_full_table_coicop.csv', index=False)
print('\n✅ Saved: outputs/inv292_full_table_coicop.csv')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 8 — CHART: GENERICS WITH DEMOGRAPHIC PROFILE BY DIVISION
# ═══════════════════════════════════════════════════════════════════════════════
gen_demo     = especificos.copy()
demo_by_div  = gen_demo.groupby('div_name').size().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(demo_by_div.index, demo_by_div.values, color='#9C27B0', alpha=0.85)
for bar, val in zip(bars, demo_by_div.values):
    ax.text(val + 0.05, bar.get_y() + bar.get_height() / 2,
            str(val), va='center', fontsize=10)

ax.set_xlabel('Number of generics with a demographic profile', fontsize=11)
ax.set_title(
    'INPC 2024 Generics with a gender or demographic reference\n'
    'The official basket aggregates them with a single national weight — the G-CPI does not',
    fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'inv292_demographic_generics.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: outputs/inv292_demographic_generics.png')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 9 — EXECUTIVE SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════
n_gender = len(especificos)
n_unchanged = sit.get('Igual', 0)
n_disagg    = sit.get('Desagregado', 0)
n_merged    = sit.get('Fusionado', 0)

print('═' * 72)
print('  EXECUTIVE SUMMARY — INVENTORY OF 292 INPC 2024 GENERICS')
print('  Relevance for the G-CPI Project')
print('═' * 72)
print(f"""
OFFICIAL SOURCE
  INEGI — Update of the CPI Basket and Weights 2024 (Aug 22, 2024)
  Weights derived from: ENIGH Seasonal 2022 ← same source as the G-CPI ✅

2024 BASKET
  {por_div['n_genericos'].sum()} generics across 13 COICOP 2018 divisions
  vs. 2018 basket: {n_unchanged} unchanged · {n_disagg} disaggregated · {n_merged} merged

KEY FINDING FOR THE G-CPI
  {n_gender} of the 292 generics have an explicit demographic profile
  (women's clothing, men's clothing, prenatal consultation, etc.)
  → The official CPI weights them with a SINGLE NATIONAL weight
  → The G-CPI reveals that those weights differ by household headship type

CONNECTION TO ARTHUR'S FEEDBACK
  Point 1 — Comparing G-CPI vs. the official index:
    Both use ENIGH 2022 as their source, but the CPI uses a single
    national basket. The gap between the G-CPI and the CPI quantifies
    the bias introduced by not differentiating by household headship.

  Point 2 — Specialised basket by group:
    This inventory is the complete map of all 292 available generics.
    A future extension would compute inflation at the generic × group level.

OUTPUT FILES IN outputs/
  inv292_by_division.csv            — generics by COICOP division
  inv292_weighted_subgenerics.csv   — sub-generics with internal weights
  inv292_full_table_coicop.csv      — all 292 generics with their division
  inv292_generics_by_division.png   — composition chart by status
  inv292_demographic_generics.png   — chart of generics with gender profile
""")
print('✅ Analysis complete.')